<a href="https://colab.research.google.com/github/ccreichenbach/AMLproject/blob/main/Hybrid_Classifier_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import BertModel
from torch.optim import AdamW

In [2]:
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: False
GPU name: No GPU


In [3]:
df = pd.read_csv('/content/drive/MyDrive/Applied Machine Learning/Final Project/fake_news_dataset.csv')
df.head()

,title,text,date,source,author,category,label
0,Foreign Democrat final.,more tax development both store agreement lawy...,2023-03-10,NY Times,Paula George,Politics,real
1,To offer down resource great point.,probably guess western behind likely next inve...,2022-05-25,Fox News,Joseph Hill,Politics,fake
2,Himself church myself carry.,them identify forward present success risk sev...,2022-09-01,CNN,Julia Robinson,Business,fake
3,You unit its should.,phone which item yard Republican safe where po...,2023-02-07,Reuters,Mr. David Foster DDS,Science,fake
4,Billion believe employee summer how.,wonder myself fact difficult course forget exa...,2023-04-03,CNN,Austin Walker,Technology,fake


In [4]:
df.describe()

,title,text,date,source,author,category,label
count,20000,20000,20000,19000,19000,20000,20000
unique,20000,20000,1096,8,17051,7,2
top,Turn present write decision town human personal.,suffer tree increase prevent organization easy...,2023-08-31,Daily News,Michael Smith,Health,fake
freq,1,1,32,2439,12,2922,10056


In [5]:
df = df[:7000]

In [6]:
df['content'] = df['title'].fillna("")+" "+df["text"].fillna("")

<ipython-input-6-1f046ac5fcfa>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['content'] = df['title'].fillna("")+" "+df["text"].fillna("")


In [7]:
source_enc = LabelEncoder()
category_enc = LabelEncoder()

In [8]:
df['source_enc'] = source_enc.fit_transform(df['source'].fillna('unknown'))
df['category_enc'] = category_enc.fit_transform(df['category'].fillna('unknown'))

<ipython-input-8-b9a1ee8cc0bf>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['source_enc'] = source_enc.fit_transform(df['source'].fillna('unknown'))
<ipython-input-8-b9a1ee8cc0bf>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['category_enc'] = category_enc.fit_transform(df['category'].fillna('unknown'))


In [9]:
#Tokenization
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens = tokenizer(
    list(df['content']),
    padding='max_length',
    max_length=512,
    truncation=True,
    return_tensors="pt"
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:
class FakeNewsDataset(Dataset):
  def __init__(self, encodings, sources, categories, labels):
    self.encodings = encodings
    self.sources = sources
    self.categories = categories
    self.labels = labels

  def __getitem__(self, idx):
    item = {key: val[idx] for key, val in self.encodings.items()}
    item['source'] = torch.tensor(self.sources[idx], dtype=torch.long)
    item['category'] = torch.tensor(self.categories[idx], dtype=torch.long)
    item['label'] = torch.tensor(self.labels[idx], dtype=torch.long)
    return item

  def __len__(self):
    return len(self.labels)

In [11]:
label_enc = LabelEncoder()
df['label'] = label_enc.fit_transform(df['label'])
df['label'] = df['label'].astype(int)

dataset = FakeNewsDataset(
    tokens,
    df['source_enc'].values,
    df['category_enc'].values,
    df['label'].values
)

## Hybrid BERT Model

In [12]:
class HybridBERTClassifier(nn.Module):
  def __init__(self, n_sources, n_categories, hidden_size=768, fusion_size = 128):
    super().__init__()
    self.bert = BertModel.from_pretrained("bert-base-uncased")

    # Embedding for categorical features
    self.source_emb = nn.Embedding(n_sources, 16)
    self.cat_emb = nn.Embedding(n_categories, 16)

    self.classifier = nn.Sequential(
        nn.Linear(hidden_size + 32, fusion_size),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(fusion_size, 2)
    )

  def forward(self, input_ids, attention_mask, source, category):
    outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
    cls_output = outputs.last_hidden_state[:, 0, :]

    source_emb = self.source_emb(source)
    cat_emb = self.cat_emb(category)

    x = torch.cat([cls_output, source_emb, cat_emb], dim=1)

    return self.classifier(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = HybridBERTClassifier(
    n_sources = len(source_enc.classes_),
    n_categories = len(category_enc.classes_)
)
model = model.to(device)

dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

optimizer = AdamW(model.parameters(), lr=5e-5)

model.train()
for epoch in range(4):
  for batch in dataloader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    source = batch["source"].to(device)
    category = batch["category"].to(device)
    labels = batch["label"].to(device)

    outputs = model(input_ids, attention_mask, source, category)
    loss = nn.CrossEntropyLoss()(outputs, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
  print(f"Epoch {epoch+1} Loss: {loss.item()}")